# Hello VLA

Welcome to your very first hands on lesson. In this notebook you will:

1. Confirm your Colab runtime is working.
2. Load a real multimodal model (SmolVLM) and ask it to describe an image.
3. Load a real Vision-Language-Action model (SmolVLA) and inspect its size.

You do not need to understand the code yet. Run each cell in order by pressing **Shift + Enter**.

## Step 1. Turn on a free GPU (optional but faster)

In the Colab menu, click **Runtime** then **Change runtime type**. Under **Hardware accelerator**, pick **T4 GPU**, then click **Save**.

If Colab says no GPU is available right now, that is fine. Everything in this notebook will still run on CPU, just slower.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Step 2. Load SmolVLM, the vision language brain of SmolVLA

A VLA is made of two halves: a **vision language model** that understands images and text, and an **action head** that turns that understanding into robot actions.

We are going to start by loading the vision language half on its own. It is called **SmolVLM**. We will give it an image and a question, and it will answer in plain English.

The installs below should take less than a minute.

In [ ]:
!pip install -q -U transformers

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests

model_id = 'HuggingFaceTB/SmolVLM-500M-Instruct'
processor = AutoProcessor.from_pretrained(model_id)
vlm = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=torch.float32).to(device)

n_params = sum(p.numel() for p in vlm.parameters())
print(f'Loaded {model_id}')
print(f'Parameters: {n_params/1e6:.1f}M')

In [ ]:
image_url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/savanna.jpg'
image = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
image.thumbnail((512, 512))
image

In [ ]:
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': 'Describe this scene in one sentence, then list three objects you can see.'}
        ]
    }
]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors='pt').to(device)

with torch.no_grad():
    out = vlm.generate(**inputs, max_new_tokens=120)

answer = processor.batch_decode(out, skip_special_tokens=True)[0]
print(answer)

Stop and notice what just happened. You gave a neural network an image plus a question in English, and it answered in English. That was not possible five years ago. This exact model, or a close cousin, is the brain of every modern VLA.

Now let's load the full VLA.

## Step 3. Load the full SmolVLA

SmolVLA is the SmolVLM above plus an **action expert**, a small transformer trained to turn the VLM's understanding into sequences of robot actions.

The install below takes two to four minutes on Colab. Be patient and do not interrupt it.

In [ ]:
!pip install -q 'lerobot[smolvla]'

In [ ]:
try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    policy = SmolVLAPolicy.from_pretrained('lerobot/smolvla_base')
    n_params = sum(p.numel() for p in policy.parameters())
    print('Loaded lerobot/smolvla_base')
    print(f'Total parameters: {n_params/1e6:.1f}M')
    print()
    print('Top level modules inside SmolVLA:')
    for name, _ in policy.named_children():
        print(' -', name)
except Exception as e:
    print('SmolVLA did not load cleanly in this environment.')
    print('That is OK for Module 0. You already saw the vision language half work above.')
    print()
    print('Error was:')
    print(repr(e))

## What you just did

1. You loaded a 500M parameter multimodal model in about one minute and asked it a question about an image. It answered in fluent English.
2. You loaded a real 450M parameter Vision-Language-Action model from the Hugging Face LeRobot team and inspected its architecture.
3. You did all of this in a browser, for free, on a machine you do not own.

You are now standing at the finish line of the course. In the next 19 modules, we will walk back to the start and build up every single piece of what you just ran, from the very basics of NumPy and gradient descent, to the Transformer, to modern robotics math, all the way back to fine tuning SmolVLA on a new robot task.

See you in Module 1.